# ARGUS Phase 2: MLM Pre-training and Anomaly Scoring on Kaggle

This notebook trains ARGUS-BERT from the verified chunked tokenized dataset, then uses the trained MLM checkpoint to score sessions with reconstruction loss. It defaults to the existing `sessions_train.pt` manifest under `/kaggle/input` and does not rerun expensive sessionization/tokenization unless you explicitly enable the optional preprocessing section.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import textwrap
import time

REPO_URL = "https://github.com/NIghtIngale340/ARGUS"
REPO_DIR = Path("/kaggle/working/argus-log-intelligence-platform")
REFRESH_REPO = True

TOKENIZED_ROOT = Path("/kaggle/input/datasets/nightingale21/argus-tokenized-58day-verified/data/tokenized")
TRAIN_MANIFEST = TOKENIZED_ROOT / "sessions_train.pt"
VAL_MANIFEST = TOKENIZED_ROOT / "sessions_val.pt"

OUTPUT_DIR = Path("/kaggle/working/argus_mlm_checkpoints")
BATCH_SIZE = 64
NUM_WORKERS = 2
MAX_STEPS = 1000
SAVE_EVERY = 100
EVAL_EVERY = 100
MAX_VAL_BATCHES = 10
LEARNING_RATE = 1e-4
WARMUP_STEPS = 100
GRADIENT_ACCUMULATION_STEPS = 1

SCORE_BATCH_SIZE = 128
SCORE_NUM_WORKERS = 2
SCORE_LIMIT_CHUNKS = 25
SCORE_OUT_CSV = Path("/kaggle/working/argus_val_scores_sample.csv")
THRESHOLDS_OUT_CSV = Path("/kaggle/working/argus_val_score_thresholds.csv")

# Use these only for quick debugging. Keep None for real training.
LIMIT_CHUNKS = None
LIMIT_SESSIONS = None

RUN_OPTIONAL_PREPROCESSING = False


## Runtime Helpers

`run_stream` prints subprocess output while it runs, so Kaggle errors are visible immediately instead of being hidden behind a generic `CalledProcessError`.


In [ ]:
def run_stream(command, cwd=None, env=None):
    print("$", " ".join(str(part) for part in command), flush=True)
    process = subprocess.Popen(
        [str(part) for part in command],
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Command failed with exit code {return_code}: {' '.join(str(part) for part in command)}")


def print_runtime_state():
    try:
        import torch
        print("torch:", torch.__version__)
        print("cuda available:", torch.cuda.is_available())
        if torch.cuda.is_available():
            print("gpu:", torch.cuda.get_device_name(0))
    except Exception as exc:
        print("torch check failed:", repr(exc))
    total, used, free = shutil.disk_usage("/kaggle/working")
    print(f"/kaggle/working free disk: {free / 1024**3:.1f} GB")


print_runtime_state()


## Clone or Refresh the Repository

Keep `REFRESH_REPO=True` when you have pushed fixes to GitHub and want Kaggle to pull the current code.


In [ ]:
if REFRESH_REPO and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    run_stream(["git", "clone", REPO_URL, str(REPO_DIR)], cwd="/kaggle/working")
else:
    run_stream(["git", "pull", "--ff-only"], cwd=REPO_DIR)

print("Repo ready:", REPO_DIR)


## Install Runtime Dependencies

Kaggle normally has PyTorch installed. This cell makes sure the lightweight project dependencies needed for training and parquet/tokenized artifact handling are present.


In [ ]:
run_stream([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "transformers>=4.37.0",
    "pyarrow>=14.0.0",
    "tqdm>=4.67.1",
    "pyyaml>=6.0",
], cwd=REPO_DIR)


## Validate Attached Tokenized Dataset

The training path expects manifests plus sibling chunk folders. The manifest is small; the tensors are in `sessions_*_chunks/chunk_*.pt`.


In [ ]:
for path in [TRAIN_MANIFEST, VAL_MANIFEST]:
    print(path, "exists=", path.exists())
    if not path.exists():
        raise FileNotFoundError(path)

for folder in [TOKENIZED_ROOT / "sessions_train_chunks", TOKENIZED_ROOT / "sessions_val_chunks"]:
    chunk_count = len(list(folder.glob("chunk_*.pt"))) if folder.exists() else 0
    print(folder, "chunks=", chunk_count)
    if chunk_count == 0:
        raise RuntimeError(f"No chunk files found under {folder}")


## Optional Preprocessing and Tokenization

This is disabled by default. Use it only when you are rebuilding artifacts from raw LANL auth logs. For Phase 2 pre-training, the verified tokenized dataset is the right default because it avoids days of duplicated preprocessing and large disk pressure.


In [ ]:
if RUN_OPTIONAL_PREPROCESSING:
    RAW_AUTH_PATH = Path("/kaggle/input/lanl-auth/auth.txt")
    OUTPUT_ROOT = Path("/kaggle/working/argus_outputs")
    if not RAW_AUTH_PATH.exists():
        raise FileNotFoundError(RAW_AUTH_PATH)

    run_stream([
        sys.executable, "scripts/build_sessions.py",
        "--input", str(RAW_AUTH_PATH),
        "--output-dir", str(OUTPUT_ROOT / "data/sessions"),
        "--start-day", "1",
        "--end-day", "58",
        "--max-tokens", "512",
        "--bucket-count", "256",
        "--bucket-workers", "1",
    ], cwd=REPO_DIR, env={**os.environ, "PYTHONPATH": str(REPO_DIR)})

    for split in ["train", "val", "test"]:
        cmd = [
            sys.executable, "scripts/build_vocab_and_tokenize.py",
            "--sessions-glob", str(OUTPUT_ROOT / "data/sessions/day_*.parquet"),
            "--split", split,
            "--max-len", "16",
            "--parquet-batch-size", "20000",
            "--tokenized-chunk-size", "20000",
            "--token-id-dtype", "int16",
            "--attention-mask-dtype", "bool",
            "--resume-tokenized",
            "--progress-log", str(OUTPUT_ROOT / f"progress_{split}.jsonl"),
            "--tokenized-out", str(OUTPUT_ROOT / f"data/tokenized/sessions_{split}.pt"),
        ]
        if split == "train":
            cmd += ["--vocab-out", str(OUTPUT_ROOT / "data/tokenized/vocab.json"), "--reuse-vocab-if-exists"]
        else:
            cmd += ["--vocab-in", str(OUTPUT_ROOT / "data/tokenized/vocab.json")]
        run_stream(cmd, cwd=REPO_DIR, env={**os.environ, "PYTHONPATH": str(REPO_DIR)})
else:
    print("Skipping preprocessing/tokenization. Using verified attached tokenized dataset.")


## One-Batch Smoke Check

Run this before real training. It verifies manifest loading, masking, compact tensor dtype handling, and the BERT forward pass.


In [ ]:
smoke_cmd = [
    sys.executable, "scripts/train_mlm_smoke.py",
    "--train-manifest", str(TRAIN_MANIFEST),
    "--limit-chunks", "1",
    "--limit-sessions", "8",
    "--batch-size", "2",
]
run_stream(smoke_cmd, cwd=REPO_DIR, env={**os.environ, "PYTHONPATH": str(REPO_DIR)})


## Phase 2 Pre-training Run

Start with `MAX_STEPS=1000`. Increase only after you confirm loss logs, checkpoint creation, GPU memory, and runtime are stable. `num_workers=2` is conservative for Kaggle; increase only if GPU is waiting on data and RAM stays healthy.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_cmd = [
    sys.executable, "-m", "src.training.pretrain",
    "--train-manifest", str(TRAIN_MANIFEST),
    "--val-manifest", str(VAL_MANIFEST),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--learning-rate", str(LEARNING_RATE),
    "--warmup-steps", str(WARMUP_STEPS),
    "--gradient-accumulation-steps", str(GRADIENT_ACCUMULATION_STEPS),
    "--max-steps", str(MAX_STEPS),
    "--save-every", str(SAVE_EVERY),
    "--eval-every", str(EVAL_EVERY),
    "--max-val-batches", str(MAX_VAL_BATCHES),
    "--log-every", "10",
    "--output-dir", str(OUTPUT_DIR),
]

if LIMIT_CHUNKS is not None:
    train_cmd += ["--limit-chunks", str(LIMIT_CHUNKS)]
if LIMIT_SESSIONS is not None:
    train_cmd += ["--limit-sessions", str(LIMIT_SESSIONS)]

run_stream(train_cmd, cwd=REPO_DIR, env={**os.environ, "PYTHONPATH": str(REPO_DIR)})


## Resume From a Checkpoint

Use this cell if Kaggle stops and you still have checkpoint files in `/kaggle/working` from the same session. For persistence across sessions, save the archive generated below as a Kaggle output or upload it elsewhere after training.


In [ ]:
checkpoints = sorted(OUTPUT_DIR.glob("checkpoint_step_*.pt"))
if not checkpoints:
    print("No checkpoints found yet.")
else:
    resume_checkpoint = checkpoints[-1]
    print("Latest checkpoint:", resume_checkpoint)
    resume_cmd = train_cmd + ["--resume-checkpoint", str(resume_checkpoint)]
    # Uncomment when you intentionally want to resume.
    # run_stream(resume_cmd, cwd=REPO_DIR, env={**os.environ, "PYTHONPATH": str(REPO_DIR)})


## Archive Checkpoints

Kaggle working storage is temporary. Archive checkpoints at the end of a useful run so they appear under notebook outputs.


In [ ]:
archive_base = Path("/kaggle/working/argus_mlm_checkpoints")
if OUTPUT_DIR.exists():
    archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_DIR)
    print("Checkpoint archive:", archive_path)
else:
    print("No checkpoint directory found:", OUTPUT_DIR)


## Phase 2.5: Choose Checkpoint for Anomaly Scoring

Use the stronger eval checkpoint when it exists. Otherwise, fall back to the newest training checkpoint. The scorer uses deterministic MLM loss: real event tokens are masked and the model's reconstruction loss becomes the anomaly score.


In [ ]:
EVAL_CHECKPOINT_DIR = Path("/kaggle/working/argus_mlm_eval_check")

eval_checkpoints = sorted(EVAL_CHECKPOINT_DIR.glob("checkpoint_step_*.pt"))
train_checkpoints = sorted(OUTPUT_DIR.glob("checkpoint_step_*.pt"))

if eval_checkpoints:
    BEST_CHECKPOINT = eval_checkpoints[-1]
elif train_checkpoints:
    BEST_CHECKPOINT = train_checkpoints[-1]
else:
    raise FileNotFoundError("No checkpoint found. Run pre-training or unzip your checkpoint archive first.")

print("Using checkpoint for scoring:", BEST_CHECKPOINT)


## Score Validation Sessions

This default scores a validation sample, not the entire validation split. `SCORE_LIMIT_CHUNKS=25` is intentionally conservative because full validation scoring can create a very large CSV. Increase it after the sample works.


In [ ]:
score_cmd = [
    sys.executable, "scripts/score_sessions.py",
    "--manifest", str(VAL_MANIFEST),
    "--checkpoint", str(BEST_CHECKPOINT),
    "--out", str(SCORE_OUT_CSV),
    "--batch-size", str(SCORE_BATCH_SIZE),
    "--num-workers", str(SCORE_NUM_WORKERS),
    "--log-every", "50000",
]

if SCORE_LIMIT_CHUNKS is not None:
    score_cmd += ["--limit-chunks", str(SCORE_LIMIT_CHUNKS)]

run_stream(score_cmd, cwd=REPO_DIR, env={**os.environ, "PYTHONPATH": str(REPO_DIR)})
print("Scores CSV:", SCORE_OUT_CSV)


## Calibrate Normal-Session Thresholds

These thresholds come from validation scores. In Phase 2.5, they are normal-baseline thresholds. Later, attack/red-team scores should land above these percentiles more often than normal traffic.


In [ ]:
import pandas as pd

scores_df = pd.read_csv(SCORE_OUT_CSV)
if scores_df.empty:
    raise RuntimeError("No scores were written; scoring did not produce a usable baseline.")

quantiles = [0.50, 0.90, 0.95, 0.99, 0.995]
threshold_rows = []
for q in quantiles:
    threshold_rows.append({
        "quantile": q,
        "threshold": float(scores_df["anomaly_score"].quantile(q)),
    })

thresholds_df = pd.DataFrame(threshold_rows)
thresholds_df.to_csv(THRESHOLDS_OUT_CSV, index=False)

print("Scored sessions:", len(scores_df))
print("Score summary:")
display(scores_df["anomaly_score"].describe())
print("Thresholds:")
display(thresholds_df)
print("Threshold CSV:", THRESHOLDS_OUT_CSV)


## Archive Phase 2.5 Outputs

Archive the score CSV and threshold CSV so they are preserved with Kaggle outputs. These are the first Phase 2 anomaly-scoring artifacts.


In [ ]:
PHASE25_DIR = Path("/kaggle/working/argus_phase25_outputs")
PHASE25_DIR.mkdir(parents=True, exist_ok=True)

for artifact in [SCORE_OUT_CSV, THRESHOLDS_OUT_CSV]:
    if artifact.exists():
        shutil.copy2(artifact, PHASE25_DIR / artifact.name)
    else:
        raise FileNotFoundError(artifact)

phase25_archive = shutil.make_archive(
    "/kaggle/working/argus_phase25_outputs",
    "zip",
    root_dir=PHASE25_DIR,
)
print("Phase 2.5 archive:", phase25_archive)
